In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

渠道统计范围

In [66]:
channel = ['零售','工程','电商']
current_date = pd.Timestamp('2025-08-31')
产品组_产品类别_map = {
    '吸油烟机':'吸油烟机',
    '灶具':'灶具',
    '烤箱':'蒸烤微',
    '蒸箱':'蒸烤微',
    '微波炉':'蒸烤微',
    '蒸烤烹饪机':'蒸烤微',
    '蒸烤微烹饪机':'蒸烤微',
    '蒸微':'蒸烤微',
    '灶消烹饪机':'灶集成',
    '灶蒸烹饪机':'灶集成',
    '灶蒸烤烹饪机':'灶集成',
    '消毒柜':'消毒柜',
    '热水器':'热水器',
    '两用炉':'热水器',
    '家用净水机':'净水机',
    '商用净水机':'净水机',
    '水槽洗碗机':'洗碗机',
    '嵌入式洗碗机':'洗碗机',
}

In [45]:
df = pd.read_excel(r'C:\Users\zhangbon\Desktop\清洗了物流-财务-渠道-产品组-国内.xlsx')
df['物料号'] = df['商品编码'].astype(str).map(lambda x: x[:13])
print(len(df))
df.head()


306178


,商品编码,渠道,实际出库数量,产品组,系统核算价,核算价,标准型号,国内/海外,物料号
0,1009001100002,工程,1,蒸烤烹饪机,3550,3550,ZK50-01-F1.i,国内,1009001100002
1,1001002100022,工程,2,吸油烟机,1508,3016,JC03A,国内,1001002100022
2,1002003400032,工程,2,灶具,900,1800,TH3B,国内,1002003400032
3,1001002000018,工程,1,吸油烟机,3668,3668,03-X1A,国内,1001002000018
4,1003000500029,工程,1,消毒柜,1550,1550,ZTD100J-J31,国内,1003000500029


In [46]:
# 长尾只看3大渠道
df1 = df.copy()
df1 = df1[df1['渠道'].isin(['零售','工程','电商'])]
df1 = df1[['物料号','渠道','产品组','标准型号','国内/海外']].drop_duplicates().reset_index(drop=True)
df1


,物料号,渠道,产品组,标准型号,国内/海外
0,1009001100002,工程,蒸烤烹饪机,ZK50-01-F1.i,国内
1,1001002100022,工程,吸油烟机,JC03A,国内
2,1002003400032,工程,灶具,TH3B,国内
3,1001002000018,工程,吸油烟机,03-X1A,国内
4,1003000500029,工程,消毒柜,ZTD100J-J31,国内
...,...,...,...,...,...
1533,1008000200075,零售,水槽洗碗机,JBSD2F-Q5S,国内
1534,1008000400008,零售,水槽洗碗机,JPSD2T-G3,国内
1535,1008000400007,零售,水槽洗碗机,JPSD2T-G3L,国内
1536,1009000900001,零售,灶蒸烤烹饪机,JZT-ZK60-X3.i,国内


In [52]:
df_product_life = pd.read_excel(r"C:\Users\zhangbon\Desktop\报告\单型号贡献\产品生命周期状态全表20250703.xlsx")
# 转换物料号列为字符串类型，并只取前13位
df_product_life[['物料号']] = df_product_life[['物料号']].astype(str).map(lambda x: x[:13])
df_product_life = df_product_life[df_product_life['物料号'].str.len()>10]
df_product_life['渠道'] = df_product_life['下属渠道']
df_product_life_map_df = df_product_life[['物料号','渠道','对应渠道状态','产品状态','产品型号','停止销售时间']]


In [57]:
df_product_life_map_df = df_product_life_map_df[df_product_life_map_df['渠道'].isin(channel)]
df_product_life_map_df = df_product_life_map_df[df_product_life_map_df['产品状态'].isin(['停止销售','停止生产'])]
# df_product_life_map_df = df_product_life_map_df[df_product_life_map_df['国内/海外']=='国内'].reset_index(drop=True)
print(df_product_life_map_df[df_product_life_map_df['停止销售时间'].isnull()])
df_product_life_map_df['停止销售时间'] = pd.to_datetime(df_product_life_map_df['停止销售时间'])
df_product_life_map_df.head()
#看一下有没有停止销售时间是空的


Empty DataFrame
Columns: [物料号, 渠道, 对应渠道状态, 产品状态, 产品型号, 停止销售时间]
Index: []


,物料号,渠道,对应渠道状态,产品状态,产品型号,停止销售时间
616,1002000500036,工程,停止销售,停止销售,C21EW,2024-10-31
683,1002000500080,电商,停止发货,停止销售,CS34BW,2022-08-03
684,1002000500080,工程,停止销售,停止销售,CS34BW,2022-08-03
685,1002000500080,零售,停止销售,停止销售,CS34BW,2022-08-03
911,1001000300073,零售,停止发货,停止销售,CXW-189-JX17S(不带罩),2020-08-27


In [63]:
df_calu = pd.merge(df1,df_product_life_map_df,how='inner',on=['物料号','渠道'])
df_calu

,物料号,渠道,产品组,标准型号,国内/海外,对应渠道状态,产品状态,产品型号,停止销售时间
0,1002001500070,工程,灶具,FZ6G,国内,停止销售,停止销售,JZT-FZ6G-12T,2024-10-31
1,1001000800337,工程,吸油烟机,EH37,国内,停止销售,停止销售,CXW-258-EH37,2022-07-07
2,1001002000004,工程,吸油烟机,X1A,国内,停止销售,停止销售,CXW-258-X1A,2025-06-19
3,1009000600012,工程,蒸烤烹饪机,ZK-TS1.i,国内,停止销售,停止销售,ZK-TS1.i,2025-04-25
4,1018000200002,工程,嵌入式洗碗机,JPCD11E-NT02,国内,停止销售,停止销售,JPCD11E-NT02,2024-07-03
...,...,...,...,...,...,...,...,...,...
263,1005000700001,零售,烤箱,KQD60F-F1,国内,停止销售,停止生产,KQD60F-F1,2024-07-12
264,1007000400001,工程,蒸箱,SCD42-F1,国内,停止发货,停止销售,SCD42-F1,2025-01-19
265,1008000200075,零售,水槽洗碗机,JBSD2F-Q5S,国内,停止销售,停止销售,JBSD2F-Q5S(不带底),2020-10-30
266,1008000400008,零售,水槽洗碗机,JPSD2T-G3,国内,停止销售,停止销售,JPSD2T-GD03,2023-01-12


In [68]:
from calendar import month
from pandas import DateOffset


for index,row in df_calu.iterrows():
    if row['渠道'] == '零售':
        if row['停止销售时间'] < current_date - pd.DateOffset(month=12):
            df_calu.loc[index,'是否长尾型号'] = '是'
    if row['渠道'] == '电商':
        if row['停止销售时间'] < current_date - pd.DateOffset(month=12):
            df_calu.loc[index,'是否长尾型号'] = '是'
    if row['渠道'] == '工程':
        if row['停止销售时间'] < current_date - pd.DateOffset(months=30):
            df_calu.loc[index,'是否长尾型号'] = '是'
df_calu['是否长尾型号'] = df_calu['是否长尾型号'].fillna('否')
df_calu['产品类别'] = df_calu['产品组'].map(产品组_产品类别_map)
df_calu

    

,物料号,渠道,产品组,标准型号,国内/海外,对应渠道状态,产品状态,产品型号,停止销售时间,是否长尾型号,产品类别
0,1002001500070,工程,灶具,FZ6G,国内,停止销售,停止销售,JZT-FZ6G-12T,2024-10-31,否,灶具
1,1001000800337,工程,吸油烟机,EH37,国内,停止销售,停止销售,CXW-258-EH37,2022-07-07,是,吸油烟机
2,1001002000004,工程,吸油烟机,X1A,国内,停止销售,停止销售,CXW-258-X1A,2025-06-19,否,吸油烟机
3,1009000600012,工程,蒸烤烹饪机,ZK-TS1.i,国内,停止销售,停止销售,ZK-TS1.i,2025-04-25,否,蒸烤微
4,1018000200002,工程,嵌入式洗碗机,JPCD11E-NT02,国内,停止销售,停止销售,JPCD11E-NT02,2024-07-03,否,洗碗机
...,...,...,...,...,...,...,...,...,...,...,...
263,1005000700001,零售,烤箱,KQD60F-F1,国内,停止销售,停止生产,KQD60F-F1,2024-07-12,是,蒸烤微
264,1007000400001,工程,蒸箱,SCD42-F1,国内,停止发货,停止销售,SCD42-F1,2025-01-19,否,蒸烤微
265,1008000200075,零售,水槽洗碗机,JBSD2F-Q5S,国内,停止销售,停止销售,JBSD2F-Q5S(不带底),2020-10-30,是,洗碗机
266,1008000400008,零售,水槽洗碗机,JPSD2T-G3,国内,停止销售,停止销售,JPSD2T-GD03,2023-01-12,是,洗碗机


In [75]:
df_out = pd.DataFrame()
df_out['产品类别'] = ['吸油烟机','灶具','蒸烤微','灶集成','消毒柜','热水器','净水机','洗碗机']
df_calu['标准型号总数'] = df_calu.groupby('产品类别')['标准型号'].transform('nunique')
df_calu['长尾标准型号数'] = df_calu[df_calu['是否长尾型号']=='是'].groupby('产品类别')['标准型号'].transform('nunique')
df_calu['长尾标准型号占比'] = df_calu['长尾标准型号数']/df_calu['标准型号总数']
df_temp = df_calu[['产品类别','长尾标准型号数','长尾标准型号占比','标准型号总数']].drop_duplicates()
df_temp = df_temp[df_temp['长尾标准型号占比']>0].reset_index(drop=True)
df_out = pd.merge(df_out,df_temp,on='产品类别',how='left')
df_out


,产品类别,长尾标准型号数,长尾标准型号占比,标准型号总数
0,吸油烟机,48.0,0.872727,55
1,灶具,19.0,0.558824,34
2,蒸烤微,23.0,0.958333,24
3,灶集成,6.0,1.000000,6
4,消毒柜,2.0,0.666667,3
5,热水器,4.0,0.800000,5
6,净水机,10.0,1.000000,10
7,洗碗机,17.0,0.739130,23
